In [1]:
import os
os.chdir("..")

In [2]:
from pathlib import Path
import json
from datasets import load_dataset

DEVICE = 'cuda'
MODEL_ID = "Qwen/QwQ-32B"
# MODEL_ID = "Qwen/Qwen2.5-32B"
model_type = "qwq"
model_name = "qwq-32b"
LAYERS = list(range(50))
N_ROWS = 40
CONT_SIZE = 100

# Initialize current directory
CUR_DIR = Path(".").absolute()

In [3]:
def load_dataset_from_file(domain_name, task_name):
    """Load dataset from a JSON file"""
    prompt_dir = CUR_DIR / Path(f"./cot-planning/results/{domain_name}/{model_name}/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

def load_datasets(domain_names, dataset_types):
    """Load datasets and prepare evaluation results"""
    task_name = "plan_generation_po"
    
    # Load evaluation results
    eval_results = [
        load_dataset_from_file(domain_name, task_name)["instances"] 
        for domain_name in domain_names
    ]
    eval_results = [{x["dataset_idx"]: x for x in er} for er in eval_results]
    
    # Load datasets
    datasets = [
        load_dataset(f"dmitriihook/{model_name}-planning-{dataset_type}")["train"]
        for dataset_type in dataset_types
    ]
    
    # Load label datasets
    # label_datasets = [
    #     load_dataset(f"dmitriihook/{domain_name.replace('_', '-')}-qwq-reasoning-parts-exploration")["train"]
    #     for domain_name in domain_names
    # ]
    
    # label_datasets = [
    #     {x["index"]: x for x in ld} for ld in label_datasets
    # ]
    
    label_datasets = [
        None for _ in domain_names
    ]
    
    return eval_results, datasets, label_datasets

In [4]:
from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN, THINK_START_TOKEN, DOMAIN_PHRASES


In [5]:
tokenizer = initialize_tokenizer(MODEL_ID)

In [6]:
domain_names = ["blocksworld_4_blocks"]
dataset_types = ["4-blocks"]

eval_results, datasets, label_datasets = load_datasets(domain_names, dataset_types)

In [7]:
item = datasets[0][0]

In [8]:
tokenized_item = tokenize_blocksworld_generation(tokenizer, item)

In [9]:
print(tokenizer.decode(tokenized_item[0, :3000]))

<|im_start|>user
I am playing with a set of blocks where I need to arrange the blocks into stacks. Here are the actions I can do

Pick up a block
Unstack a block from on top of another block
Put down a block
Stack a block on top of another block

I have the following restrictions on my actions:
I can only pick up or unstack one block at a time.
I can only pick up or unstack a block if my hand is empty.
I can only pick up a block if the block is on the table and the block is clear. A block is clear if the block has no other blocks on top of it and if the block is not picked up.
I can only unstack a block from on top of another block if the block I am unstacking was really on top of the other block.
I can only unstack a block from on top of another block if the block I am unstacking is clear.
Once I pick up or unstack a block, I am holding the block.
I can only put down a block that I am holding.
I can only stack a block on top of another block if I am holding the block being stacked.
I 